# Image Captioning: CNN Encoder + Transformer Decoder

## Teaching a Model to Describe What It Sees

In this notebook we combine two things you already know:
- **CNN feature extraction** (from our ResNet fine-tuning project)
- **Transformer decoding** (from our NMT and GPT2 notebooks)

The architecture is simple: a pretrained ResNet extracts visual features
from an image (the "encoder"), and a Transformer decoder generates a
text caption one word at a time, using **cross-attention** to look at
the relevant parts of the image at each step.

### The architecture

```
Image → ResNet (frozen) → spatial feature grid → [Transformer Decoder] → "a dog playing in the park"
                                                       ↑
                                              cross-attention
                                         (same as NMT decoder → encoder)
```

### Connections to what we've built

| NMT Transformer | Image Captioning |
|---|---|
| Encoder: Transformer self-attention over source words | Encoder: ResNet CNN over image pixels |
| Encoder output: one vector per source token | Encoder output: one vector per spatial region |
| Decoder cross-attention: "which source words matter?" | Decoder cross-attention: "which image regions matter?" |
| Output: Spanish words | Output: English caption words |

The decoder is **identical** to the NMT decoder. It doesn't know or care
that the encoder features come from an image rather than a sentence.
Cross-attention just sees a sequence of key-value vectors.

In [ ]:
# uncomment if needed
!pip install datasets pillow -q
#!pip install transformers -q

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import time
import os

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

---
## 1. Load the Flickr8k Dataset

Flickr8k contains 8,000 images, each with **5 human-written captions**.
The captions describe what's happening in the image in plain English.

This is like our NMT dataset, but instead of (source sentence, target sentence)
pairs, we have (image, caption) pairs.

The decoder's job is the same in both cases: generate English text,
one token at a time, using cross-attention to look at the "source"
(a sentence in NMT, an image here).


In [ ]:
# uncomment if using Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset
import datasets

print("Loading Flickr8k dataset...")
ds = load_dataset("jxie/flickr8k", cache_dir = "content/drive/MyDrive/Colab/datasets")
print(ds)

# Explore the data
print(f"\nTrain examples: {len(ds['train'])}")
print(f"Test examples:  {len(ds['test'])}")

# Show a sample
sample = ds['train'][0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Sample caption: {sample['caption_0']}")

In [ ]:
from tqdm.auto import tqdm

new_train = []
new_test = []

# Each image in Flickr8k has 5 different captions (caption_0 through caption_4).
# We want to expand the dataset so each (image, caption) pair is a SEPARATE
# training example. If we have 8,000 images with 5 captions each, we should
# end up with 40,000 training examples.
#
# Q: Why is this better than just using caption_0 for each image?
#    What does the model learn from seeing 5 different descriptions of the same image?
#
# Your answer:
#

for x in tqdm(ds['train'], desc="Processing train captions"):
    image = x['image']

    # TODO: Create 5 separate training examples from this one image.
    # Each example should be a dict with keys "image" and "caption".
    # The captions are in x['caption_0'] through x['caption_4'].
    # Append all 5 to new_train.
    #
    # Hint: This is about 5 lines of code. Each line creates a dict
    #       and appends it to new_train.
    pass

# For test, we only keep one caption per image (for evaluation)
for x in tqdm(ds['test'], desc="Processing test captions"):
    image = x['image']
    y0 = {"image": image, "caption": x['caption_0']}
    new_test.append(y0)

ds['train'] = datasets.Dataset.from_list(new_train)
ds['test'] = datasets.Dataset.from_list(new_test)

print(ds)
print(f"\nExpected ~40,000 train examples (8,000 images × 5 captions)")


In [ ]:

# Explore the data
print(f"\nTrain examples: {len(ds['train'])}")
print(f"Test examples:  {len(ds['test'])}")

# Show a sample
sample = ds['train'][0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Sample caption: {sample['caption']}")

In [ ]:
# Visualize some examples
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, ax in enumerate(axes.flatten()):
    sample = ds['train'][i * 100]
    ax.imshow(sample['image'])
    caption = sample['caption']
    # Wrap long captions
    wrapped = '\n'.join([caption[j:j+40] for j in range(0, len(caption), 40)])
    ax.set_title(wrapped, fontsize=9)
    ax.axis('off')
plt.suptitle('Flickr8k: Images with Human Captions', fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Build a Vocabulary

Use the GPT2 Tokenizer so unknown words can be processed

In [ ]:
import re
from transformers import GPT2Tokenizer

class Vocabulary:
    """Word-level vocabulary using GPT2Tokenizer."""
    def __init__(self, min_freq=3, tokenizer_name='gpt2'):
        self.tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_name)


        self.tokenizer.add_special_tokens({
            'pad_token': '<pad>',
            'bos_token': '<sos>',
            'unk_token': '<unk>',
        })

        self.PAD = self.tokenizer.pad_token_id   # 50257 or 50258
        self.SOS = self.tokenizer.bos_token_id   # distinct from EOS
        self.EOS = self.tokenizer.eos_token_id   # 50256
        self.UNK = self.tokenizer.unk_token_id
        self.SPECIAL = [self.PAD, self.SOS, self.EOS, self.UNK]

        # These are kept for compatibility but will primarily use tokenizer's internal vocab
        self.idx2word = list(self.tokenizer.get_vocab().keys())
        self.word2idx = self.tokenizer.get_vocab()
        self.min_freq = min_freq # min_freq might not be directly used by pre-trained tokenizer

    def build(self, captions):
        # For GPT2Tokenizer, vocab is already built from pretrained model.
        # We only need to ensure special tokens are correctly registered.
        # This method is kept for compatibility with the original flow.
        return self

    def clean_caption(self, caption):
        """Lowercase and remove punctuation except apostrophes. (GPT2 tokenizer handles most)"""
        caption = caption.lower().strip()
        # GPT2 tokenizer handles a lot of normalization, but some basic cleaning helps.
        caption = re.sub(r"[^a-z0-9\s']", '', caption)
        caption = re.sub(r'\s+', ' ', caption).strip()
        return caption

    def encode(self, caption, max_len=30):
        cleaned_caption = self.clean_caption(caption)
        # Tokenize, add SOS/EOS manually for consistency with original model logic
        tokens = [self.SOS] + self.tokenizer.encode(cleaned_caption, add_special_tokens=False)
        if len(tokens) >= max_len:
            tokens = tokens[:max_len - 1] + [self.EOS]
        else:
            tokens.append(self.EOS)
        return tokens

    def decode(self, ids):
        # Filter out special tokens for decoding, then use tokenizer's decode
        filtered_ids = [token_id for token_id in ids if token_id not in self.SPECIAL]
        return self.tokenizer.decode(filtered_ids, skip_special_tokens=True).strip()

    def __len__(self):
        return len(self.tokenizer)

# Build vocabulary from training captions
# The build method for GPT2Tokenizer just ensures special tokens are set up.
all_captions = [ex['caption'] for ex in ds['train']]
vocab = Vocabulary(min_freq=3).build(all_captions)

print(f"Vocabulary size: {len(vocab):,}")
print(f"Sample encode: '{all_captions[0]}'")
encoded = vocab.encode(all_captions[0])
print(f"  → {encoded}")
print(f"  → '{vocab.decode(encoded)}'")
print(f"PAD={vocab.PAD}, SOS={vocab.SOS}, EOS={vocab.EOS}, UNK={vocab.UNK}")

---
## 3. Image Encoder: Pretrained ResNet

This is the same pretrained ResNet from your fine-tuning project, but
used differently. Instead of taking the final classification output
(a single vector), we take the **spatial feature map** from the last
convolutional layer — before the global average pooling.

For ResNet-50, this gives us a (7 × 7 × 2048) feature map.
We reshape it to (49, 2048) — 49 spatial positions, each with a
2048-dimensional feature vector. These 49 vectors play the same role
as the encoder hidden states h₁...hₙ in our NMT model, except each
one represents a region of the image rather than a word.

```
NMT:   [h₁, h₂, ..., h₂₀]     — one vector per source word
Image: [r₁, r₂, ..., r₄₉]     — one vector per 32×32 pixel region
```

The decoder's cross-attention doesn't know the difference — it just
attends to a sequence of key-value vectors.

In [ ]:
class ImageEncoder(nn.Module):
    """
    Pretrained ResNet-50 as an image encoder.

    We use ResNet as a feature extractor, but differently from our
    classification project. There, we used the FINAL output — one vector
    per image (after global average pooling). Here, we take the SPATIAL
    feature map from the last convolutional layer — BEFORE pooling.

    ResNet-50's last conv layer outputs: (batch, 2048, 7, 7)
        2048 = feature channels
        7×7  = spatial grid (the image has been downsampled to 7×7)

    We reshape this to: (batch, 49, 2048)
        49 = 7×7 spatial positions, flattened into a sequence
        2048 = feature dimension at each position

    Then project 2048 → d_model so it matches our decoder's dimension.

    Final output: (batch, 49, d_model)
        This is a sequence of 49 vectors — one per image region.
        The decoder will cross-attend to these, just like it
        cross-attended to encoder hidden states in NMT.
    """
    def __init__(self, d_model):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Remove the final avgpool and fc layers — we want the spatial features
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # Freeze the CNN — just like we froze ResNet during fine-tuning
        for param in self.backbone.parameters():
            param.requires_grad = False

        # TODO: Define a linear projection from 2048 → d_model
        # You will use a pytorch Linear layer
        # This is needed because ResNet produces 2048-dim features,
        # but our Transformer decoder expects d_model-dim vectors.
        #
        # Q: Why can't we just set d_model=2048 instead of adding this projection?
        #    (Hint: think about what d_model=2048 would do to the decoder's parameter count)
        #
        # Your answer:
        #
        self.projection = ???

    def forward(self, images):
        # images: (batch, 3, 224, 224)

        # Run through frozen ResNet backbone
        with torch.no_grad():
            features = self.backbone(images)   # (batch, 2048, 7, 7)

        # Reshape from image grid to sequence:
        # (batch, 2048, 7, 7) → (batch, 2048, 49) → (batch, 49, 2048)
        batch_size = features.size(0)
        features = features.view(batch_size, 2048, -1).permute(0, 2, 1)

        # Project to d_model dimensions
        features = self.projection(features)   # (batch, 49, d_model)

        return features


# Verify
_test_enc = ImageEncoder(d_model=256)
_test_img = torch.randn(2, 3, 224, 224)
_test_out = _test_enc(_test_img)
print(f"Input:  {_test_img.shape}")      # (2, 3, 224, 224)
print(f"Output: {_test_out.shape}")      # (2, 49, 256)
print(f"\n49 = 7×7 spatial regions, each with a {256}-dim feature vector")
print(f"This is like having 49 'encoder hidden states' for the decoder.")
del _test_enc, _test_img, _test_out


---
## 4. Transformer Decoder (from NMT)

This is the **same decoder architecture** from our NMT notebook.
Three sublayers per block:
1. **Masked self-attention** — caption attends to itself (previous words)
2. **Cross-attention** — caption attends to image features
3. **Feed-forward** — processes each position independently

The cross-attention sublayer is the bridge between vision and language.
Remember how it works:
- **Query** comes from the decoder (the caption being generated)
- **Key** and **Value** come from the encoder (the image features)
- The dot product Q·Kᵀ asks: "which image regions are relevant for generating this word?"
- The weighted sum over V extracts information from those regions

In NMT: Q=decoder state, K=V=source sentence hidden states

Here:   Q=decoder state, K=V=image spatial features

The decoder doesn't know the difference. Cross-attention just sees vectors.


In [ ]:
# These are copied directly from our NMT notebook — no changes needed

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = K.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        #TODO: Define the weight matrices
        self.W_q = #???
        self.W_k = #???
        self.W_v = #???
        self.W_o = #???

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)


        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.d_model)
        return self.W_o(attn_output), attn_weights


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [ ]:
class CaptionDecoderLayer(nn.Module):
    """
    This is our NMT DecoderLayer, reused for image captioning.
    Three sublayers:
      1. Masked self-attention (caption tokens attend to previous caption tokens)
      2. Cross-attention (caption tokens attend to image regions)
      3. Feed-forward

    In NMT, cross-attention connected: decoder state → source sentence
    Here, cross-attention connects:     decoder state → image features
    Same mechanism. Different modalities.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, tgt_mask):
        # x:              (batch, caption_len, d_model) — decoder's caption representations
        # encoder_output: (batch, 49, d_model)          — image region features from ResNet

        # 1. Masked self-attention over caption tokens
        #    Q=K=V=x — each caption token attends to all previous caption tokens
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))

        # 2. Cross-attention: caption attends to image
        #
        # TODO: Fill in the three arguments to cross_attn.
        #
        # Remember from our NMT notebook:
        #   cross_attn(query, key, value)
        #   - query comes from the DECODER (the thing asking "what should I look at?")
        #   - key and value come from the ENCODER (the thing being looked at)
        #
        # Here the decoder is generating a caption, and the encoder produced image features.
        # Which variable is the query? Which is key and value?
        #
        attn_out, _ = self.cross_attn(???, ???, ???)
        x = self.norm2(x + self.dropout2(attn_out))

        # 3. Feed-forward
        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_out))

        return x


---
## 5. Full Captioning Model

Putting it together:

```
Image → ResNet encoder → spatial features (49, d_model)
                                ↓ (cross-attention K, V)
Caption → word embedding + positional encoding → decoder layers → linear → vocabulary logits
```

Compare to our NMT Transformer:
```
Source → embedding + pos_enc → encoder layers → encoder output
                                     ↓ (cross-attention K, V)
Target → embedding + pos_enc → decoder layers → linear → vocabulary logits
```

The only structural change: we replaced the Transformer encoder with a CNN.

In [ ]:
class ImageCaptioner(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=3,
                 d_ff=512, max_caption_len=30, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_caption_len = max_caption_len

        self.encoder = ImageEncoder(d_model)

        self.word_embedding = nn.Embedding(vocab_size, d_model, padding_idx=vocab.PAD)
        self.pos_embedding = nn.Embedding(max_caption_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.decoder_layers = nn.ModuleList(
            [CaptionDecoderLayer(d_model, num_heads, d_ff, dropout)
             for _ in range(num_layers)]
        )

        self.norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size)

        mask = torch.tril(torch.ones(max_caption_len, max_caption_len))
        self.register_buffer('causal_mask', mask.view(1, 1, max_caption_len, max_caption_len))

    def encode_image(self, images):
        return self.encoder(images)

    def decode(self, captions, encoder_output):
        B, T = captions.shape
        tok = self.word_embedding(captions) * math.sqrt(self.d_model)
        pos = self.pos_embedding(torch.arange(T, device=captions.device))
        x = self.dropout(tok + pos)
        tgt_mask = self.causal_mask[:, :, :T, :T]
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, tgt_mask)
        x = self.norm(x)
        logits = self.output_projection(x)
        return logits

    def forward(self, images, captions):
        encoder_output = self.encode_image(images)
        logits = self.decode(captions, encoder_output)
        return logits

    @torch.no_grad()
    def generate(self, images, max_len=30, sos_idx=50256, eos_idx=50256,
                 temperature=1.0, top_k=None, min_p=None):
        """
        Autoregressive generation using current vocab IDs, with optional sampling strategies.
        Args:
            images: Batch of image tensors (batch_size, 3, H, W)
            max_len: Maximum length of the generated caption.
            sos_idx: Start-of-sequence token ID.
            eos_idx: End-of-sequence token ID.
            temperature: Softmax temperature. Higher values make the distribution flatter. (Default: 1.0)
            top_k: If not None, keep only the top_k most likely tokens for sampling. (Default: None)
            min_p: If not None, keep only tokens whose cumulative probability is less than or equal to min_p. (Default: None)
        Returns:
            Generated caption token IDs (batch_size, generated_length).
        """
        self.eval()
        encoder_output = self.encode_image(images) # (batch_size, 49, d_model)
        batch_size = images.size(0)
        generated = torch.full((batch_size, 1), sos_idx, dtype=torch.long, device=images.device) # (batch_size, 1)

        for _ in range(max_len - 1):
            # Decode up to the current generated length
            logits = self.decode(generated, encoder_output) # (batch_size, current_seq_len, vocab_size)
            next_token_logits = logits[:, -1, :]           # (batch_size, vocab_size) - logits for the next token

            # Apply temperature scaling
            if temperature != 1.0:
                next_token_logits = next_token_logits / temperature

            # --- Sampling Logic per batch item ---
            next_tokens_batch = []
            for i in range(batch_size):
                logits_row = next_token_logits[i] # Logits for current batch item (vocab_size,)

                # Top-K filtering
                if top_k is not None and top_k > 0:
                    # Get the top_k values and their indices
                    values, _ = torch.topk(logits_row, top_k)
                    # Set logits of tokens outside the top_k to a very small number (-inf)
                    logits_row[logits_row < values[-1]] = float('-inf')

                # Min-P (Top-P) filtering
                if min_p is not None and min_p > 0.0 and min_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits_row, descending=True)
                    # Convert logits to probabilities (only for sorting and cumulative sum)
                    sorted_probs = F.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                    # Find the index where cumulative probability first exceeds min_p
                    # All tokens from this index onward will be removed
                    indices_to_remove = cumulative_probs > min_p
                    # Shift the indices_to_remove to the right to keep the first token that made it exceed min_p
                    if indices_to_remove.any():
                        first_index_to_remove = torch.nonzero(indices_to_remove, as_tuple=True)[0][0]
                        logits_row[sorted_indices[first_index_to_remove:]] = float('-inf')


                # After filtering, sample from the modified distribution
                probs = F.softmax(logits_row, dim=-1) # Re-normalize probabilities after filtering

                # Ensure numerical stability if all probs became 0 (e.g., aggressive filtering)
                if probs.sum() == 0:
                    # Fallback: if all options filtered out, predict EOS
                    next_token_single = torch.tensor([eos_idx], device=images.device)
                else:
                    next_token_single = torch.multinomial(probs, num_samples=1)

                next_tokens_batch.append(next_token_single)

            next_token = torch.stack(next_tokens_batch).view(batch_size, 1) # (batch_size, 1)

            generated = torch.cat([generated, next_token], dim=1)

            # Early stopping check (original logic for batch_size=1)
            if (next_token == eos_idx).any() and batch_size == 1:
                break

        return generated

In [ ]:
# Hyperparameters
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 3
D_FF = 512
MAX_CAPTION_LEN = 30
DROPOUT = 0.25

model = ImageCaptioner(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_caption_len=MAX_CAPTION_LEN,
    dropout=DROPOUT
).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,} (decoder + projection)")
print(f"Frozen parameters:    {frozen:,} (ResNet backbone)")
print(f"\nThe ResNet is frozen — we only train the decoder.")
print(f"Same strategy as fine-tuning: reuse pretrained features.")

### OPTIONAL Suggested Improved Hyperparameters
Increasing the depth and width of the Transformer decoder typically yields the best quality gains. Note that this will increase training time and memory usage.

In [ ]:
# Improved Hyperparameters for better quality
# THIS WILL SLOW EVERYTHING DOWN

D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 6  # Deeper decoder for better reasoning
D_FF = 2048     # 4x d_model is standard
MAX_CAPTION_LEN = 30
DROPOUT = 0.25

model = ImageCaptioner(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_caption_len=MAX_CAPTION_LEN,
    dropout=DROPOUT
).to(device)

print(f"New Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## 6. Data Pipeline

We need to:
1. Preprocess images for ResNet (resize, normalize)
2. Tokenize and pad captions
3. Batch them together

We also apply data augmentation by adding random image tweaks to the image on each batch during training. There is also a weird-but-normal "ResNet Normalization" step

In [ ]:
from torch.utils.data import Dataset, DataLoader

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class CaptionDataset(Dataset):
    def __init__(self, hf_dataset, vocab, transform, max_len=30):
        self.data = hf_dataset
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Process image
        image = item['image'].convert('RGB')
        image = self.transform(image)

        # Process caption
        caption = item['caption']
        tokens = self.vocab.encode(caption, self.max_len)
        tokens += [self.vocab.PAD] * (self.max_len - len(tokens))
        tokens = torch.tensor(tokens, dtype=torch.long)

        return image, tokens


BATCH_SIZE = 320

#TODO: Fill in the missing parameters

train_dataset = CaptionDataset(ds['train'], vocab, ???, MAX_CAPTION_LEN)
test_dataset = CaptionDataset(ds['test'], vocab, ???, MAX_CAPTION_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

# Verify
img_batch, cap_batch = next(iter(train_loader))
print(f"Image batch: {img_batch.shape}")     # (32, 3, 224, 224)
print(f"Caption batch: {cap_batch.shape}")   # (32, 30)
print(f"\nSample caption tokens: {cap_batch[0].tolist()}")
print(f"Decoded: '{vocab.decode(cap_batch[0].tolist())}'")

---
## 7. Training

Same training pattern as NMT: **teacher forcing** with shifted targets.

During training, we give the decoder the *ground truth* caption as input
(not its own predictions). The input is shifted by one position relative
to the target — at each position, the model sees the previous words and
must predict the next word.

```
Full caption:  [<sos>,  a,   dog,  runs,  <eos>, <pad>]
Decoder input: [<sos>,  a,   dog,  runs,  <eos>       ]  ← sees this
Target:        [a,      dog, runs, <eos>, <pad>        ]  ← predicts this
```

The causal mask ensures position k can only attend to positions 0..k,
so the model can't peek at future words even though the full caption
is provided during training.


In [ ]:
LEARNING_RATE = 1e-4

# TODO: Create the optimizer.
#
# Important: We froze the ResNet backbone — those parameters should NOT
# be updated. Only the decoder and projection layers should be trained.
#
#
# Only optimize trainable parameters (decoder + projection, not ResNet)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for images, captions in loader:
        images = images.to(device)
        captions = captions.to(device)

        # Teacher forcing: the decoder sees the ground truth caption as input,
        # shifted by one position. This is the same trick from our NMT notebook.
        #
        # If the full caption is: [<sos>, a, dog, runs, <eos>, <pad>, <pad>]
        #
        # TODO: Create cap_input and cap_target from captions.
        #
        # cap_input  should be everything EXCEPT the last token
        #            → the decoder SEES this
        # cap_target should be everything EXCEPT the first token
        #            → the decoder should PREDICT this
        #
        # Use slicing: captions[:, ???]
        #
        cap_input = ???
        cap_target = ???

        logits = model(images, cap_input)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            cap_target.reshape(-1),
            ignore_index=vocab.PAD,
            label_smoothing=0.1
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), 1.0
        )
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    for images, captions in loader:
        images = images.to(device)
        captions = captions.to(device)
        cap_input = captions[:, :-1]
        cap_target = captions[:, 1:]

        logits = model(images, cap_input)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            cap_target.reshape(-1),
            ignore_index=vocab.PAD
        )
        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
#uncomment if in google
#from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define checkpoint directory
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab/captions_2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory created at: {CHECKPOINT_DIR}")

# Define path for saving the best model
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_image_captioning_model.pth')


train_losses = []
test_losses = []
NUM_EPOCHS = 15
START_EPOCH = 1
PATIENCE = 5
patience_count = 0

best_test_loss = float('inf')

print(f"Training for {NUM_EPOCHS} epochs...")
print(f"  {len(train_loader)} batches/epoch, batch_size={BATCH_SIZE}")
print()

start_time = time.time()

for epoch in range(START_EPOCH, NUM_EPOCHS + START_EPOCH):
    train_loss = train_epoch(model, train_loader, optimizer)
    test_loss = evaluate(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    scheduler.step(test_loss)

    elapsed = time.time() - start_time
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train: {train_loss:.4f} | Test: {test_loss:.4f} | "
          f"LR: {lr:.2e} | {elapsed:.0f}s")

    # Save the model if it has the best test loss so far
    if test_loss < best_test_loss:
        patience_count = 0
        best_test_loss = test_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_test_loss,
        }, BEST_MODEL_PATH)
        print(f"  --> New best model saved with test loss: {best_test_loss:.4f}")
    else:
      patience_count += 1
      if patience_count >= PATIENCE:
        print(f"  --> Early stopping triggered. Best test loss: {best_test_loss:.4f}")
        break

print(f"\nDone in {(time.time()-start_time)/60:.1f} minutes")

---

---
## 8. Generate Captions

Let's see what the model produces for test images.

In [ ]:
# Load the best model for inference
import random
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded from epoch {checkpoint['epoch']} with test loss: {checkpoint['loss']:.4f}")

In [ ]:
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

def show_caption(model, dataset, idx):
    model.eval()
    image_tensor, caption_tokens = dataset[idx]
    image_tensor = image_tensor.unsqueeze(0).to(device)

    # Use IDs from the vocab instance
    generated_ids = model.generate(image_tensor, max_len=MAX_CAPTION_LEN, sos_idx=vocab.SOS, eos_idx=vocab.EOS, temperature=0.3)
    generated_caption = vocab.decode(generated_ids[0].tolist())

    gt_caption = vocab.decode(caption_tokens.tolist())

    img_display = inv_normalize(image_tensor.squeeze(0).cpu())
    img_display = img_display.permute(1, 2, 0).clamp(0, 1).numpy()

    plt.figure(figsize=(6, 6))
    plt.imshow(img_display)
    plt.title(f"Generated: {generated_caption}\nGround truth: {gt_caption}", fontsize=10)
    plt.axis('off')
    plt.show()

for j in range(10):
    i = random.randint(0, len(test_dataset) - 1)
    show_caption(model, test_dataset, i)

---
## 9. Visualize Cross-Attention: Where Does the Model Look?

This is the payoff. The cross-attention weights tell us **which image
regions** the model focuses on when generating each word.

We extract the cross-attention weights from the last decoder layer,
average across heads, and reshape the 49 attention weights back to
the 7×7 spatial grid. Then we overlay this as a heatmap on the image.

This is exactly like the Bahdanau attention heatmaps from our NMT
notebook — but instead of showing which source *words* the decoder
attends to, it shows which image *regions* the decoder looks at.

In [ ]:
def get_attention_maps(model, image_tensor):
    """
    Generate a caption and extract cross-attention weights at each step.
    Returns the generated caption and attention maps for each word.
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)

    # Encode image once
    encoder_output = model.encode_image(image_tensor)

    # Generate step by step, capturing attention at each step
    generated = [vocab.SOS]
    attention_maps = []

    for step in range(MAX_CAPTION_LEN - 1):
        tgt = torch.tensor([generated], dtype=torch.long, device=device)
        T = tgt.size(1)

        # Run through decoder manually to capture cross-attention
        tok = model.word_embedding(tgt) * math.sqrt(model.d_model)
        pos = model.pos_embedding(torch.arange(T, device=device))
        x = model.dropout(tok + pos)

        tgt_mask = model.causal_mask[:, :, :T, :T]

        cross_attn_weights = None
        for layer in model.decoder_layers:
            # Self-attention
            attn_out, _ = layer.self_attn(x, x, x, tgt_mask)
            x = layer.norm1(x + layer.dropout1(attn_out))

            # Cross-attention — capture these weights
            attn_out, cross_attn_weights = layer.cross_attn(
                x, encoder_output, encoder_output
            )
            x = layer.norm2(x + layer.dropout2(attn_out))

            # Feed-forward
            ff_out = layer.feed_forward(x)
            x = layer.norm3(x + layer.dropout3(ff_out))

        x = model.norm(x)
        logits = model.output_projection(x)
        next_token = logits[0, -1, :].argmax().item()

        # Save attention for this step (average across heads, last position)
        # cross_attn_weights: (1, num_heads, T, 49)
        attn = cross_attn_weights[0, :, -1, :].mean(dim=0)  # (49,)
        attention_maps.append(attn.cpu().detach().numpy())

        generated.append(next_token)
        if next_token == vocab.EOS:
            break

    return generated, attention_maps

In [ ]:
def visualize_attention(model, dataset, idx):
    """
    Show the image with attention heatmaps for each generated word.
    """
    image_tensor, gt_tokens = dataset[idx]

    # Get the raw image for display
    img_display = inv_normalize(image_tensor).permute(1, 2, 0).clamp(0, 1).numpy()

    # Generate caption with attention maps
    generated_ids, attention_maps = get_attention_maps(model, image_tensor)

    # Use the tokenizer to convert IDs to words correctly
    # skip <sos> at index 0
    words = []
    for idx_val in generated_ids[1:]:
        if idx_val == vocab.EOS:
            break
        # Use the tokenizer's method to get the string representation
        word = vocab.tokenizer.decode([idx_val]).strip()
        words.append(word)

    # Trim attention maps to match words
    attention_maps = attention_maps[:len(words)]

    if len(words) == 0:
        print("Model generated empty caption.")
        return

    # Plot: original image + attention for each word
    n_words = min(len(words), 8)  # show at most 8 words
    fig, axes = plt.subplots(1, n_words + 1, figsize=(3 * (n_words + 1), 3.5))

    # Original image
    axes[0].imshow(img_display)
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    # Attention heatmap for each word
    for i in range(n_words):
        ax = axes[i + 1]
        ax.imshow(img_display)

        # Reshape 49 attention weights to 7x7 and upscale to 224x224
        attn_map = attention_maps[i].reshape(7, 7)
        attn_map = np.array(Image.fromarray(attn_map).resize((224, 224), Image.BILINEAR))

        ax.imshow(attn_map, alpha=0.6, cmap='jet')
        ax.set_title(f'"{words[i]}"', fontsize=10, fontweight='bold')
        ax.axis('off')

    caption = ' '.join(words)
    gt = vocab.decode(gt_tokens.tolist())
    fig.suptitle(f'Generated: "{caption}"\nGround truth: "{gt}"',
                 fontsize=11, y=1.05)
    plt.tight_layout()
    plt.show()

# Visualize attention for several test images
for i in range(5):
    idx = random.randint(0, len(test_dataset) - 1)
    visualize_attention(model, test_dataset, idx)
    print()

---
## 10. Individual Attention Heads

Just like in our NMT notebook, different heads may specialize.
Some might focus on objects, others on backgrounds or spatial layout.

In [ ]:
def visualize_heads(model, dataset, idx, word_idx=0):
    """
    Show all attention heads for a specific word in the generated caption.
    """
    image_tensor, gt_tokens = dataset[idx]
    img_display = inv_normalize(image_tensor).permute(1, 2, 0).clamp(0, 1).numpy()

    # Forward pass to get per-head attention
    model.eval()
    image_input = image_tensor.unsqueeze(0).to(device)
    encoder_output = model.encode_image(image_input)

    generated_ids, _ = get_attention_maps(model, image_tensor)
    words = []
    for id_val in generated_ids[1:]:
        if id_val == vocab.EOS:
            break
        # Use the tokenizer's method to get the string representation
        word = vocab.tokenizer.decode([id_val]).strip()
        words.append(word)

    # Re-run decoder to get per-head weights at the target step
    tgt = torch.tensor([generated_ids[:word_idx + 2]], dtype=torch.long, device=device)
    T = tgt.size(1)
    tok = model.word_embedding(tgt) * math.sqrt(model.d_model)
    pos = model.pos_embedding(torch.arange(T, device=device))
    x = model.dropout(tok + pos)
    tgt_mask = model.causal_mask[:, :, :T, :T]

    cross_w = None
    for layer in model.decoder_layers:
        attn_out, _ = layer.self_attn(x, x, x, tgt_mask)
        x = layer.norm1(x + layer.dropout1(attn_out))
        attn_out, cross_w = layer.cross_attn(x, encoder_output, encoder_output)
        x = layer.norm2(x + layer.dropout2(attn_out))
        ff_out = layer.feed_forward(x)
        x = layer.norm3(x + layer.dropout3(ff_out))

    # cross_w: (1, num_heads, T, 49)
    per_head = cross_w[0, :, -1, :].detach().cpu().numpy()  # (num_heads, 49)

    target_word = words[word_idx] if word_idx < len(words) else '?'

    fig, axes = plt.subplots(2, NUM_HEADS // 2, figsize=(3 * NUM_HEADS // 2, 7))
    for h, ax in enumerate(axes.flatten()):
        ax.imshow(img_display)
        attn_map = per_head[h].reshape(7, 7)
        attn_map = np.array(Image.fromarray(attn_map).resize((224, 224), Image.BILINEAR))
        ax.imshow(attn_map, alpha=0.6, cmap='jet')
        ax.set_title(f'Head {h}', fontsize=10)
        ax.axis('off')

    fig.suptitle(f'All {NUM_HEADS} heads attending while generating "{target_word}"\n'\
                 f'Full caption: "{" ".join(words)}"', fontsize=12)
    plt.tight_layout()
    plt.show()

# Visualize heads for a few examples
for i in range(2):
    idx = random.randint(0, len(test_dataset) - 1)
    visualize_heads(model, test_dataset, idx, word_idx=1)

---
## 11. Discussion

### What we built

A multimodal model — one that processes *both* images and text. The image
encoder (ResNet) and text decoder (Transformer) are completely different
architectures, connected by cross-attention. The cross-attention mechanism
is the universal bridge: it doesn't care what produced the key-value
vectors, only that they're in the right shape.

### Connections across the semester

| Concept | Where we first saw it | How it appears here |
|---|---|---|
| CNN feature extraction | ResNet fine-tuning project | Image encoder (frozen ResNet) |
| Cross-attention | Bahdanau attention in NMT | Decoder attending to image regions |
| Causal masking | NMT Transformer decoder | Caption generation (same mask) |
| Teacher forcing | Seq2seq RNN training | Training with ground truth captions |
| Transfer learning | ResNet fine-tuning | Frozen CNN + trained decoder |
| Attention visualization | NMT heatmaps | Heatmaps over image regions |

### The path to modern vision-language models

Our architecture (CNN + Transformer decoder) is the foundation of models like:

- **Show, Attend and Tell** (Xu et al., 2015) — same idea with LSTM decoder
- **BLIP / BLIP-2** (Salesforce) — more sophisticated bridge between vision and language
- **GPT-4V / Claude's vision** — same cross-attention principle at massive scale,
  with Vision Transformers (ViT) replacing CNNs as the image encoder

The core mechanism is always the same: encode the image into a sequence of
feature vectors, and let a language model attend to them via cross-attention.

### Questions to consider

1. The attention heatmaps show where the model "looks" when generating each word.
   Does it look at sensible regions? When generating "dog", does it attend to the dog?

2. Our image encoder produces 49 spatial regions (7×7). What would happen with
   more regions (14×14 = 196)? What's the tradeoff?

3. We froze the ResNet and only trained the decoder. What would happen if we
   unfroze the last few ResNet layers? (Think about our fine-tuning experiments.)

4. Could we use transformers and run this in reverse? And generate images?

5. Modern models like GPT-4V use a Vision Transformer (ViT) instead of ResNet
   as the image encoder. ViT processes images using self-attention — so the
   entire model (encoder and decoder) would be pure Transformer. Why might
   that be advantageous?